In [1]:
import os
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz
from delta import configure_spark_with_delta_pip

# Lendo variáveis do ambiente do container
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
SPARK_MASTER = os.getenv("SPARK_MASTER", "spark://spark-master:7077")

# Define Delta Lake version compatible with your Spark
DELTA_VERSION = "3.2.0"

builder = (
    SparkSession.builder
    .appName("base_pagamento")
    .master(SPARK_MASTER)
    
    # Add Delta Lake packages explicitly
    .config("spark.jars.packages", f"io.delta:delta-spark_2.12:{DELTA_VERSION},io.delta:delta-storage:{DELTA_VERSION}")
    
    # Delta Lake SQL extensions
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    
    # MinIO / S3
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    
    # Additional Delta configs for S3
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore")
    .config("spark.sql.parquet.compression.codec", "snappy")
    
    # Optional: Hadoop AWS configuration
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
)

# Configure with Delta pip
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-00584122-a276-43af-9a87-c4d3affdc424;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 242ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

In [2]:
agora=datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc=agora.strftime("%Y%m%d%H%M%S")

In [3]:
path = "s3a://bronze/book_pagamento/"
df_book_pagamento = spark.read.parquet(path)
df_book_pagamento.show(20, truncate=False)

26/01/01 16:57:34 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/01/01 16:57:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+--------------------

In [4]:
df_book_pagamento.createOrReplaceTempView("raw_00")

In [5]:
df_book_pagamento.count()

21829628

In [6]:
raw_00_com_safra = spark.sql("""
    SELECT
        *,
        CAST(
            date_format(
                to_timestamp(DAT_CRIACAO_DW, 'ddMMMyyyy:HH:mm:ss'),
                'yyyyMM'
            ) AS INT
        ) AS SAFRA
    FROM raw_00
""")

raw_00_com_safra.createOrReplaceTempView("raw_00_com_safra")
raw_00_com_safra.cache()

In [7]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos,
        COUNT(DISTINCT CONTRATO) as contrato_distintos,
        COUNT(DISTINCT DW_NUM_CLIENTE) as num_tel_distintos
    FROM raw_00_com_safra
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(20, truncate=False)

+------+------------+-------------+------------------+-----------------+
|SAFRA |total_linhas|cpf_distintos|contrato_distintos|num_tel_distintos|
+------+------------+-------------+------------------+-----------------+
|202310|820565      |476221       |541727            |541813           |
|202311|833427      |483153       |549229            |549324           |
|202312|979497      |533700       |610263            |610539           |
|202401|889710      |515472       |588888            |589053           |
|202402|915370      |526350       |601108            |601291           |
|202403|1054435     |572971       |657923            |658176           |
|202404|970518      |558386       |641180            |641321           |
|202405|1037571     |584157       |672982            |673121           |
|202406|1024408     |584739       |674906            |675117           |
|202407|1042481     |596388       |690442            |690590           |
|202408|1084153     |612231       |711084          

In [8]:
def contagem_percentual(coluna: str):
    query = f"""
        WITH total AS (
            SELECT COUNT(*) AS total_registros
            FROM raw_00_com_safra
        )
        SELECT
            r.{coluna}                               AS valor_coluna,
            COUNT(*)                                AS qtd_registros,
            ROUND(
                COUNT(*) * 100.0 / t.total_registros,
                2
            )                                        AS pct_registros
        FROM raw_00_com_safra r
        CROSS JOIN total t
        GROUP BY r.{coluna}, t.total_registros
        ORDER BY qtd_registros DESC
    """
    return spark.sql(query)

In [9]:
df_resultado = contagem_percentual("DAT_ATUALIZACAO_ATIVIDADE")
df_resultado.show(truncate=False)

+------------------+-------------+-------------+
|valor_coluna      |qtd_registros|pct_registros|
+------------------+-------------+-------------+
|NULL              |20730830     |94.97        |
|18FEB2025:18:54:47|238          |0.00         |
|18FEB2025:18:54:52|225          |0.00         |
|18FEB2025:18:54:51|218          |0.00         |
|18FEB2025:18:54:50|204          |0.00         |
|18FEB2025:18:54:48|186          |0.00         |
|12MAR2025:04:19:30|182          |0.00         |
|18FEB2025:18:54:49|169          |0.00         |
|08FEB2025:05:25:18|167          |0.00         |
|08JAN2025:03:40:08|167          |0.00         |
|08JAN2025:03:40:01|164          |0.00         |
|08JAN2025:03:40:17|164          |0.00         |
|12MAR2025:04:19:34|163          |0.00         |
|12MAR2025:04:19:39|159          |0.00         |
|12MAR2025:04:19:42|158          |0.00         |
|12MAR2025:04:19:33|158          |0.00         |
|18FEB2025:18:54:46|158          |0.00         |
|12MAR2025:04:19:43|

In [10]:
df_resultado = contagem_percentual("COD_FUNDO_ATIVIDADE")
df_resultado.show(truncate=False)

+------------+-------------+-------------+
|valor_coluna|qtd_registros|pct_registros|
+------------+-------------+-------------+
|NULL        |21819036     |99.95        |
|127291234   |1395         |0.01         |
|112486948   |1029         |0.00         |
|870146020   |834          |0.00         |
|102816296   |454          |0.00         |
|835361356   |402          |0.00         |
|208728872   |255          |0.00         |
|745991108   |237          |0.00         |
|772940148   |136          |0.00         |
|106673111   |119          |0.00         |
|106400699   |117          |0.00         |
|939540591   |107          |0.00         |
|118426118   |89           |0.00         |
|102995928   |59           |0.00         |
|963449181   |49           |0.00         |
|166813901   |43           |0.00         |
|136024547   |43           |0.00         |
|811302811   |40           |0.00         |
|825289776   |40           |0.00         |
|163375953   |38           |0.00         |
+----------

In [11]:
df_resultado = contagem_percentual("COD_METODO_PAGAMENTO")
df_resultado.show(truncate=False)

+------------+-------------+-------------+
|valor_coluna|qtd_registros|pct_registros|
+------------+-------------+-------------+
|NULL        |16297184     |74.66        |
|1           |1657841      |7.59         |
|3           |1538749      |7.05         |
|5           |1401465      |6.42         |
|4           |647567       |2.97         |
|2           |285169       |1.31         |
|6           |1653         |0.01         |
+------------+-------------+-------------+



In [12]:
df_resultado = contagem_percentual("COD_NETUNO_PAGAMENTO")
df_resultado.show(truncate=False)

+------------+-------------+-------------+
|valor_coluna|qtd_registros|pct_registros|
+------------+-------------+-------------+
|NULL        |21829628     |100.00       |
+------------+-------------+-------------+



In [13]:
df_resultado = contagem_percentual("COD_DESALOCACAO_CREDITO")
df_resultado.show(truncate=False)

+------------+-------------+-------------+
|valor_coluna|qtd_registros|pct_registros|
+------------+-------------+-------------+
|NULL        |21829628     |100.00       |
+------------+-------------+-------------+



In [14]:
print('lista de colunas para tipar')
for col in spark.table("raw_00_com_safra").columns:
    print('try_cast(' + col + ' as) as ' + col + ',')

lista de colunas para tipar
try_cast(NUM_CPF as) as NUM_CPF,
try_cast(DAT_STATUS_FATURA as) as DAT_STATUS_FATURA,
try_cast(CONTRATO as) as CONTRATO,
try_cast(SEQ_FATURA as) as SEQ_FATURA,
try_cast(NUM_SUB_SEQ_FATURA as) as NUM_SUB_SEQ_FATURA,
try_cast(NUM_CREDITO_SEQ as) as NUM_CREDITO_SEQ,
try_cast(DW_TIPO_FATURA as) as DW_TIPO_FATURA,
try_cast(IND_STATUS_FATURA as) as IND_STATUS_FATURA,
try_cast(DW_NUM_CLIENTE as) as DW_NUM_CLIENTE,
try_cast(DW_AREA as) as DW_AREA,
try_cast(DW_UN_NEGOCIO as) as DW_UN_NEGOCIO,
try_cast(DW_FORMA_PAGAMENTO as) as DW_FORMA_PAGAMENTO,
try_cast(VAL_PAGAMENTO_FATURA as) as VAL_PAGAMENTO_FATURA,
try_cast(DAT_CRIACAO_DW as) as DAT_CRIACAO_DW,
try_cast(DW_BANCO as) as DW_BANCO,
try_cast(DW_TIPO_PAGAMENTO as) as DW_TIPO_PAGAMENTO,
try_cast(NUM_BANCO_PAGAMENTO as) as NUM_BANCO_PAGAMENTO,
try_cast(NUM_AGENCIA_PAGAMENTO as) as NUM_AGENCIA_PAGAMENTO,
try_cast(NUM_CC_PAGAMENTO as) as NUM_CC_PAGAMENTO,
try_cast(DW_MOTIVO_ESTORNO as) as DW_MOTIVO_ESTORNO,
try_cast(VAL

In [15]:
lake = spark.sql(     
    """
        select
        
            -- campos do arquivo --

            try_cast(NUM_CPF as STRING) as NUM_CPF,
            try_cast(SAFRA as INT) as SAFRA,
            case 
                when trim(DAT_STATUS_FATURA) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_STATUS_FATURA), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_STATUS_FATURA,
            try_cast(CONTRATO as BIGINT) as CONTRATO,
            try_cast(SEQ_FATURA as INT) as SEQ_FATURA,
            try_cast(NUM_SUB_SEQ_FATURA as INT) as NUM_SUB_SEQ_FATURA,
            try_cast(NUM_CREDITO_SEQ as INT) as NUM_CREDITO_SEQ,
            try_cast(DW_TIPO_FATURA as INT) as DW_TIPO_FATURA,
            try_cast(IND_STATUS_FATURA as STRING) as IND_STATUS_FATURA,
            try_cast(DW_NUM_CLIENTE as STRING) as DW_NUM_CLIENTE,
            try_cast(DW_AREA as INT) as DW_AREA,
            try_cast(DW_UN_NEGOCIO as INT) as DW_UN_NEGOCIO,
            try_cast(DW_FORMA_PAGAMENTO as INT) as DW_FORMA_PAGAMENTO,
            try_cast(VAL_PAGAMENTO_FATURA as DECIMAL(10,2)) as VAL_PAGAMENTO_FATURA,
            case 
                when trim(DAT_CRIACAO_DW) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_DW), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_DW,
            try_cast(DW_BANCO as INT) as DW_BANCO,
            try_cast(DW_TIPO_PAGAMENTO as INT) as DW_TIPO_PAGAMENTO,
            try_cast(NUM_BANCO_PAGAMENTO as STRING) as NUM_BANCO_PAGAMENTO,
            try_cast(NUM_AGENCIA_PAGAMENTO as STRING) as NUM_AGENCIA_PAGAMENTO,
            try_cast(NUM_CC_PAGAMENTO as STRING) as NUM_CC_PAGAMENTO,
            try_cast(DW_MOTIVO_ESTORNO as INT) as DW_MOTIVO_ESTORNO,
            try_cast(VAL_DESCONTO_ITEM as DECIMAL(10,2)) as VAL_DESCONTO_ITEM,
            try_cast(VAL_PAGAMENTO_ITEM as DECIMAL(10,2)) as VAL_PAGAMENTO_ITEM,
            try_cast(VAL_JUROS_MULTAS_ITEM as DECIMAL(10,2)) as VAL_JUROS_MULTAS_ITEM,
            try_cast(VAL_MULTA_EQUIP_ITEM as DECIMAL(10,2)) as VAL_MULTA_EQUIP_ITEM,
            try_cast(VAL_MULTA_EQUIP_TOTAL as DECIMAL(10,2)) as VAL_MULTA_EQUIP_TOTAL,
            try_cast(VAL_MULTA_FID_ITEM as DECIMAL(10,2)) as VAL_MULTA_FID_ITEM,
            try_cast(COD_ORIGEM_NETUNO as STRING) as COD_ORIGEM_NETUNO,
            try_cast(COD_CONTA_ATIVIDADE as STRING) as COD_CONTA_ATIVIDADE,
            try_cast(SEQ_ENTIDADE_ATIVIDADE as INT) as SEQ_ENTIDADE_ATIVIDADE,
            case 
                when trim(DAT_CRIACAO_ATIVIDADE) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_ATIVIDADE), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_ATIVIDADE,
            case 
                when trim(DAT_ATUALIZACAO_ATIVIDADE) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_ATUALIZACAO_ATIVIDADE), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ATUALIZACAO_ATIVIDADE,
            try_cast(COD_LOGIN_OPERADOR_ATIVIDADE as STRING) as COD_LOGIN_OPERADOR_ATIVIDADE,
            try_cast(COD_ATIVIDADE as STRING) as COD_ATIVIDADE,
            try_cast(COD_RAZAO_ATIVIDADE as STRING) as COD_RAZAO_ATIVIDADE,
            case 
                when trim(DAT_BAIXA_ATIVIDADE) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_BAIXA_ATIVIDADE), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_BAIXA_ATIVIDADE,
            try_cast(VAL_BAIXA_ATIVIDADE as DECIMAL(10,2)) as VAL_BAIXA_ATIVIDADE,
            case 
                when trim(DAT_DEPOSITO_ATIVIDADE) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_DEPOSITO_ATIVIDADE), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_DEPOSITO_ATIVIDADE,
            try_cast(COD_FUNDO_ATIVIDADE as STRING) as COD_FUNDO_ATIVIDADE,
            try_cast(COD_BANCO_ATIVIDADE as STRING) as COD_BANCO_ATIVIDADE,
            try_cast(NUM_CONTA_ATIVIDADE as STRING) as NUM_CONTA_ATIVIDADE,
            try_cast(COD_AGENCIA_ATIVIDADE as STRING) as COD_AGENCIA_ATIVIDADE,
            try_cast(SEQ_ENTIDADE_PAGAMENTO as INT) as SEQ_ENTIDADE_PAGAMENTO,
            case 
                when trim(DAT_CRIACAO_PAGAMENTO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_PAGAMENTO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_PAGAMENTO,
            case 
                when trim(DAT_ATUALIZACAO_PAGAMENTO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_ATUALIZACAO_PAGAMENTO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ATUALIZACAO_PAGAMENTO,
            try_cast(COD_LOGIN_PAGAMENTO as STRING) as COD_LOGIN_PAGAMENTO,
            try_cast(COD_FORMA_PAGAMENTO as STRING) as COD_FORMA_PAGAMENTO,
            try_cast(VAL_ORIGINAL_PAGAMENTO as DECIMAL(10,2)) as VAL_ORIGINAL_PAGAMENTO,
            try_cast(NUM_FATURA_PAGAMENTO as STRING) as NUM_FATURA_PAGAMENTO,
            try_cast(COD_TIPO_PAGAMENTO as STRING) as COD_TIPO_PAGAMENTO,
            try_cast(DSC_NOME_BANCO_PAGAMENTO as STRING) as DSC_NOME_BANCO_PAGAMENTO,
            try_cast(SEQ_ARQUIVO_PAGAMENTO as INT) as SEQ_ARQUIVO_PAGAMENTO,
            try_cast(NUM_PARCELA_PAGAMENTO as STRING) as NUM_PARCELA_PAGAMENTO,
            try_cast(NUM_AGRUPADOR_PAGAMENTO as STRING) as NUM_AGRUPADOR_PAGAMENTO,
            try_cast(DSC_PAGAMENTO as STRING) as DSC_PAGAMENTO,
            try_cast(VAL_ATUAL_PAGAMENTO as DECIMAL(10,2)) as VAL_ATUAL_PAGAMENTO,
            try_cast(COD_METODO_PAGAMENTO as INT) as COD_METODO_PAGAMENTO,
            try_cast(IND_STATUS_PAGAMENTO as STRING) as IND_STATUS_PAGAMENTO,
            case 
                when trim(DAT_STATUS_PAGAMENTO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_STATUS_PAGAMENTO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_STATUS_PAGAMENTO,
            try_cast(COD_ARQUIVO_PAGAMENTO as STRING) as COD_ARQUIVO_PAGAMENTO,
            try_cast(COD_NETUNO_PAGAMENTO as STRING) as COD_NETUNO_PAGAMENTO,
            case 
                when trim(DAT_CRIACAO_CREDITO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_CRIACAO_CREDITO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_CRIACAO_CREDITO,
            case 
                when trim(DAT_ATUALIZACAO_CREDITO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_ATUALIZACAO_CREDITO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ATUALIZACAO_CREDITO,
            try_cast(COD_LOGIN_CREDITO as STRING) as COD_LOGIN_CREDITO,
            try_cast(VAL_PAGAMENTO_CREDITO as DECIMAL(10,2)) as VAL_PAGAMENTO_CREDITO,
            try_cast(IND_TIPO_CREDITO as STRING) as IND_TIPO_CREDITO,
            try_cast(SEQ_PAGAMENTO_CREDITO as INT) as SEQ_PAGAMENTO_CREDITO,
            try_cast(SEQ_FATURA_CREDITO as INT) as SEQ_FATURA_CREDITO,
            try_cast(COD_ALOCACAO_CREDITO as STRING) as COD_ALOCACAO_CREDITO,
            try_cast(COD_DESALOCACAO_CREDITO as STRING) as COD_DESALOCACAO_CREDITO,
            try_cast(SEQ_ENTIDADE_CREDITO as INT) as SEQ_ENTIDADE_CREDITO,
            try_cast(COD_TIPO_FATURA as STRING) as COD_TIPO_FATURA,
            case 
                when trim(DAT_ATIVIDADE_CREDITO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_ATIVIDADE_CREDITO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_ATIVIDADE_CREDITO,
            case 
                when trim(DAT_VENCIMENTO_CREDITO) in ('null', 'NULL', '') then null
                else try_cast(to_timestamp(trim(DAT_VENCIMENTO_CREDITO), 'ddMMMyyyy:HH:mm:ss') as TIMESTAMP)
            end as DAT_VENCIMENTO_CREDITO,
            {pdthproc} as DATPROC

        from
            raw_00_com_safra
            
    """.format(pdthproc=dthproc))
lake.createOrReplaceTempView("lake")
lake.cache()
lake.count()  

21829628

In [16]:
lake.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- SAFRA: integer (nullable = true)
 |-- DAT_STATUS_FATURA: timestamp (nullable = true)
 |-- CONTRATO: long (nullable = true)
 |-- SEQ_FATURA: integer (nullable = true)
 |-- NUM_SUB_SEQ_FATURA: integer (nullable = true)
 |-- NUM_CREDITO_SEQ: integer (nullable = true)
 |-- DW_TIPO_FATURA: integer (nullable = true)
 |-- IND_STATUS_FATURA: string (nullable = true)
 |-- DW_NUM_CLIENTE: string (nullable = true)
 |-- DW_AREA: integer (nullable = true)
 |-- DW_UN_NEGOCIO: integer (nullable = true)
 |-- DW_FORMA_PAGAMENTO: integer (nullable = true)
 |-- VAL_PAGAMENTO_FATURA: decimal(10,2) (nullable = true)
 |-- DAT_CRIACAO_DW: timestamp (nullable = true)
 |-- DW_BANCO: integer (nullable = true)
 |-- DW_TIPO_PAGAMENTO: integer (nullable = true)
 |-- NUM_BANCO_PAGAMENTO: string (nullable = true)
 |-- NUM_AGENCIA_PAGAMENTO: string (nullable = true)
 |-- NUM_CC_PAGAMENTO: string (nullable = true)
 |-- DW_MOTIVO_ESTORNO: integer (nullable = true)
 |-- V

In [17]:
# Deduplicação caso aconteça de reprocessar mesma base
lake_dedup = spark.sql("""
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY NUM_CPF, DAT_CRIACAO_DW, CONTRATO, NUM_SUB_SEQ_FATURA
                ORDER BY DATPROC DESC
            ) AS rn
        FROM lake
    ) t
    WHERE rn = 1
""")
lake_dedup.createOrReplaceTempView("lake_dedup")
lake_dedup = spark.sql("SELECT * FROM lake_dedup")
lake_dedup.cache()
lake_dedup.count()  

21829628

In [18]:
contagem_safras = spark.sql("""
    SELECT 
        SAFRA,
        COUNT(*) as total_linhas,
        COUNT(DISTINCT NUM_CPF) as cpf_distintos,
        COUNT(DISTINCT CONTRATO) as contrato_distintos,
        COUNT(DISTINCT DW_NUM_CLIENTE) as num_tel_distintos
    FROM lake_dedup
    GROUP BY SAFRA
    ORDER BY SAFRA
""")

contagem_safras.show(20, truncate=False)

+------+------------+-------------+------------------+-----------------+
|SAFRA |total_linhas|cpf_distintos|contrato_distintos|num_tel_distintos|
+------+------------+-------------+------------------+-----------------+
|202310|820565      |476221       |541727            |541813           |
|202311|833427      |483153       |549229            |549324           |
|202312|979497      |533700       |610263            |610539           |
|202401|889710      |515472       |588888            |589053           |
|202402|915370      |526350       |601108            |601291           |
|202403|1054435     |572971       |657923            |658176           |
|202404|970518      |558386       |641180            |641321           |
|202405|1037571     |584157       |672982            |673121           |
|202406|1024408     |584739       |674906            |675117           |
|202407|1042481     |596388       |690442            |690590           |
|202408|1084153     |612231       |711084          

In [19]:
from delta.tables import DeltaTable

silver_path = "s3a://silver/base_pagamento/"

# Se a tabela ainda não existir, cria do zero
if not DeltaTable.isDeltaTable(spark, silver_path):
    print("Tabela silver não existe. Criando...")

    (
        lake_dedup
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("SAFRA")
        .save(silver_path)
    )
    print("Dados inseridos com sucesso...")
else:
    print("Tabela silver existe. Fazendo MERGE incremental...")

    delta_silver = DeltaTable.forPath(spark, silver_path)

    (
        delta_silver.alias("t")
        .merge(
            lake_dedup.alias("s"),
            """
            t.NUM_CPF = s.NUM_CPF
            AND t.DAT_STATUS_FATURA = s.DAT_STATUS_FATURA
            AND t.CONTRATO = s.CONTRATO
            AND t.NUM_SUB_SEQ_FATURA = s.NUM_SUB_SEQ_FATURA
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Dados inseridos com sucesso...")

Tabela silver não existe. Criando...


Dados inseridos com sucesso...


In [20]:
name = "base_pagamento"

df_controle = spark.sql("""
    SELECT
        '{name_table}'        AS nome_tabela,
        SAFRA                 AS safra,
        COUNT(*)              AS qtd_registros,
        current_timestamp()   AS datproc
    FROM lake_dedup
    GROUP BY SAFRA
    ORDER BY SAFRA
""".format(name_table=name))

df_controle.show()

+--------------+------+-------------+--------------------+
|   nome_tabela| safra|qtd_registros|             datproc|
+--------------+------+-------------+--------------------+
|base_pagamento|202310|       820565|2026-01-01 17:17:...|
|base_pagamento|202311|       833427|2026-01-01 17:17:...|
|base_pagamento|202312|       979497|2026-01-01 17:17:...|
|base_pagamento|202401|       889710|2026-01-01 17:17:...|
|base_pagamento|202402|       915370|2026-01-01 17:17:...|
|base_pagamento|202403|      1054435|2026-01-01 17:17:...|
|base_pagamento|202404|       970518|2026-01-01 17:17:...|
|base_pagamento|202405|      1037571|2026-01-01 17:17:...|
|base_pagamento|202406|      1024408|2026-01-01 17:17:...|
|base_pagamento|202407|      1042481|2026-01-01 17:17:...|
|base_pagamento|202408|      1084153|2026-01-01 17:17:...|
|base_pagamento|202409|      1068778|2026-01-01 17:17:...|
|base_pagamento|202410|      1077559|2026-01-01 17:17:...|
|base_pagamento|202411|      1172431|2026-01-01 17:17:..

In [21]:
silver_controle_path = "s3a://silver/controle/"
if not DeltaTable.isDeltaTable(spark, silver_controle_path):
    print("Tabela de controle não existe. Criando...")

    (
        df_controle
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_controle_path)
    )
    print("Dados inseridos com sucesso...")
else:
    print("Tabela de controle existe. Inserindo novo registro...")

    (
        df_controle
        .write
        .format("delta")
        .mode("append")
        .save(silver_controle_path)
    )
    print("Dados inseridos com sucesso...")

Tabela de controle existe. Inserindo novo registro...


Dados inseridos com sucesso...


In [22]:
spark.stop()